# UEFA Euro 2020 Final — Data Pipeline
**Italy vs England** | Wembley Stadium, London | 11 July 2021

Italy 1–1 England (AET) — **Italy won 3–2 on penalties**

Goals:
- 2' Luke Shaw (England)
- 67' Leonardo Bonucci (Italy)

This notebook fetches events from StatsBomb open data, cleans the data, and caches to `cache/`.
Run `02_visualizations.ipynb` for all figures.

**Note:** UEFA Euro 2020 includes 360-degree freeze-frame data in StatsBomb open data.

## 0. Setup

In [1]:
!pip install statsbombpy pandas numpy -q

## 1. Libraries

In [2]:
import os, io, time, warnings, json
import numpy as np
import pandas as pd
from statsbombpy import sb
warnings.filterwarnings('ignore')

CACHE_DIR = r'D:\Masaüstü\Projects\MatchAnalysis\Italy-England2020\cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache dir: {CACHE_DIR}')

Cache dir: D:\Masaüstü\Projects\MatchAnalysis\Italy-England2020\cache


## 2. Fetch Data

In [3]:
def _fetch_with_retry(fn, *args, **kwargs):
    """Retries on rate-limit errors."""
    for attempt in range(6):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f'  Error: {e}  |  waiting {wait}s... ({attempt+1}/6)')
            time.sleep(wait)
    raise RuntimeError('API unreachable after 6 attempts.')

# Competitions
cache_comps = os.path.join(CACHE_DIR, 'competitions.json')
if os.path.exists(cache_comps):
    comps = pd.read_json(cache_comps)
    print('[CACHE] competitions loaded.')
else:
    comps = _fetch_with_retry(sb.competitions)
    comps.to_json(cache_comps)
    print('[API]   competitions fetched and cached.')

euro = comps[comps['competition_name'] == 'UEFA Euro']
print(euro[['competition_id', 'season_id', 'season_name']].to_string())

[CACHE] competitions loaded.
    competition_id  season_id season_name
73              55        282        2024
74              55         43        2020


In [4]:
# UEFA Euro 2020  (season_name='2020', season_id=43)
row_2020 = euro[euro['season_name'] == '2020'].iloc[0]
competition_id = int(row_2020['competition_id'])   # 55
season_id      = int(row_2020['season_id'])         # 43
print(f'Competition ID: {competition_id}  |  Season ID: {season_id}')

# Matches
cache_matches = os.path.join(CACHE_DIR, f'matches_{competition_id}_{season_id}.json')
if os.path.exists(cache_matches):
    matches = pd.read_json(cache_matches)
    print('[CACHE] matches loaded.')
else:
    matches = _fetch_with_retry(sb.matches, competition_id=competition_id, season_id=season_id)
    matches.to_json(cache_matches)
    print('[API]   matches fetched and cached.')

print(f'Total matches: {len(matches)}')

Competition ID: 55  |  Season ID: 43
[CACHE] matches loaded.
Total matches: 51


In [5]:
# Find the Italy vs England final
mask = (
    matches['home_team'].str.contains('Italy',   case=False, na=False) |
    matches['away_team'].str.contains('Italy',   case=False, na=False)
) & (
    matches['home_team'].str.contains('England', case=False, na=False) |
    matches['away_team'].str.contains('England', case=False, na=False)
)
final_row = matches[mask].iloc[0]
match_id  = int(final_row['match_id'])

print(f'Match ID : {match_id}')
print(f'Result   : {final_row["home_team"]}  {final_row["home_score"]} - '
      f'{final_row["away_score"]}  {final_row["away_team"]}')
print(f'Date     : {final_row["match_date"]}')

Match ID : 3795506
Result   : Italy  1 - 1  England
Date     : 2021-07-11


In [6]:
# Events
cache_events = os.path.join(CACHE_DIR, f'events_{match_id}.json')
if os.path.exists(cache_events):
    events_raw = pd.read_json(cache_events)
    print('[CACHE] events loaded.')
else:
    events_raw = _fetch_with_retry(sb.events, match_id=match_id)
    events_raw.to_json(cache_events)
    print('[API]   events fetched and cached.')

print(f'Total events : {len(events_raw)}')
print(f'Columns      : {len(events_raw.columns)}')
print(events_raw['type'].value_counts().to_string())
print(f'\nPeriods in data: {sorted(events_raw["period"].unique())}')
print('Periods 3 & 4 = extra time; period 5 = penalty shootout')

[CACHE] events loaded.
Total events : 4796
Columns      : 97
type
Pass                 1350
Ball Receipt*        1317
Carry                1106
Pressure              370
Ball Recovery          93
Duel                   92
Dribble                51
Foul Committed         47
Goal Keeper            42
Foul Won               42
Clearance              39
Interception           35
Shot                   35
Block                  35
Dribbled Past          34
Miscontrol             28
Dispossessed           27
Substitution           11
Half Start             10
Half End               10
Tactical Shift          7
Injury Stoppage         5
Player Off              3
Player On               3
Starting XI             2
Referee Ball-Drop       2

Periods in data: [1, 2, 3, 4, 5]
Periods 3 & 4 = extra time; period 5 = penalty shootout


In [7]:
# 360 / Freeze-frame data (available for Euro 2020)
cache_frames = os.path.join(CACHE_DIR, f'frames_{match_id}.json')
has_360 = False
try:
    if os.path.exists(cache_frames):
        frames_raw = pd.read_json(cache_frames)
        print(f'[CACHE] 360 frames loaded. shape={frames_raw.shape}')
    else:
        frames_raw = _fetch_with_retry(sb.frames, match_id=match_id, fmt='dataframe')
        frames_raw.to_json(cache_frames)
        print(f'[API]   360 frames fetched. shape={frames_raw.shape}')
    has_360 = len(frames_raw) > 0
except Exception as e:
    print(f'360 frames not available: {e}')
    frames_raw = pd.DataFrame()
    has_360 = False

print(f'360 data available: {has_360}')

[CACHE] 360 frames loaded. shape=(58906, 7)
360 data available: True


## 3. Pre-processing

In [8]:
events = events_raw.copy()

# location -> x, y
events['x'] = events['location'].apply(lambda v: v[0] if isinstance(v, list) else np.nan)
events['y'] = events['location'].apply(lambda v: v[1] if isinstance(v, list) else np.nan)

# pass end location
if 'pass_end_location' in events.columns:
    events['pass_end_x'] = events['pass_end_location'].apply(
        lambda v: v[0] if isinstance(v, list) else np.nan)
    events['pass_end_y'] = events['pass_end_location'].apply(
        lambda v: v[1] if isinstance(v, list) else np.nan)

# carry end location
if 'carry_end_location' in events.columns:
    events['carry_end_x'] = events['carry_end_location'].apply(
        lambda v: v[0] if isinstance(v, list) else np.nan)
    events['carry_end_y'] = events['carry_end_location'].apply(
        lambda v: v[1] if isinstance(v, list) else np.nan)

# Team names
teams = events['team'].dropna().unique().tolist()
ITA   = next(t for t in teams if 'Italy'   in t)
ENG   = next(t for t in teams if 'England' in t)

print(f'Teams: "{ITA}"  vs  "{ENG}"')
print(f'x column non-null: {events["x"].notna().sum()}')

Teams: "Italy"  vs  "England"
x column non-null: 4745


## 4. Match Statistics

In [9]:
# Goals — regulation + extra time (periods 1-4); period 5 = penalties (excluded)
goals_df = events[(events['type']=='Shot') & (events['shot_outcome']=='Goal') &
                  (events['period'] <= 4)].copy()
print('=== GOALS (reg + ET) ===')
print(goals_df[['player','team','minute','period']].to_string(index=False))
print('\nNote: Italy won 3-2 on penalties (period 5 not shown here)')

=== GOALS (reg + ET) ===
          player    team  minute  period
       Luke Shaw England       1       1
Leonardo Bonucci   Italy      66       2

Note: Italy won 3-2 on penalties (period 5 not shown here)


In [10]:
def team_stats(ev, team):
    te       = ev[(ev['team'] == team) & (ev['period'] <= 4)]
    passes   = te[te['type'] == 'Pass']
    shots    = te[te['type'] == 'Shot']
    goals    = shots[shots['shot_outcome'] == 'Goal']
    dribbles = te[te['type'] == 'Dribble']
    presses  = te[te['type'] == 'Pressure']
    return {
        'Passes'       : len(passes),
        'Pass Acc (%)' : round(100 * passes['pass_outcome'].isna().sum() / max(len(passes), 1), 1),
        'Shots'        : len(shots),
        'Goals'        : len(goals),
        'Dribbles'     : len(dribbles),
        'Pressures'    : len(presses),
        'Total Events' : len(te),
    }

stats = pd.DataFrame(
    [team_stats(events, ITA), team_stats(events, ENG)],
    index=[ITA, ENG]).T
print('=== MATCH STATS (reg + ET) ===')
print(stats.to_string())

=== MATCH STATS (reg + ET) ===
               Italy  England
Passes         885.0    465.0
Pass Acc (%)    88.2     75.5
Shots           19.0      6.0
Goals            1.0      1.0
Dribbles        25.0     26.0
Pressures      157.0    213.0
Total Events  2958.0   1814.0


In [11]:
print(f'=== TOP PLAYERS — {ITA} ===')
ita_counts = events[events['team']==ITA].groupby('player').size().sort_values(ascending=False)
print(ita_counts.head(10).to_string())
print(f'\n=== TOP PLAYERS — {ENG} ===')
eng_counts = events[events['team']==ENG].groupby('player').size().sort_values(ascending=False)
print(eng_counts.head(10).to_string())

=== TOP PLAYERS — Italy ===
player
Marco Verratti                 382
Leonardo Bonucci               373
Jorge Luiz Frello Filho        347
Giorgio Chiellini              326
Giovanni Di Lorenzo            252
Lorenzo Insigne                237
Emerson Palmieri dos Santos    218
Nicolò Barella                 146
Federico Chiesa                133
Bryan Cristante                118

=== TOP PLAYERS — England ===
player
Luke Shaw          199
Kalvin Phillips    198
Harry Maguire      191
John Stones        164
Harry Kane         161
Kyle Walker        136
Declan Rice        128
Mason Mount        125
Raheem Sterling    119
Jordan Pickford    115


## 5. Save Processed Data

In [12]:
meta = {
    'ITA': ITA, 'ENG': ENG,
    'match_id': match_id,
    'competition_id': competition_id,
    'season_id': season_id,
    'has_360': has_360,
}
with open(os.path.join(CACHE_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False)

pkl_path = os.path.join(CACHE_DIR, 'events_processed.pkl')
events.to_pickle(pkl_path)

print(f'events_processed.pkl saved  ({os.path.getsize(pkl_path)//1024} KB)')
print('meta.json saved')
print(f'has_360 = {has_360}')
print('\nRun 02_visualizations.ipynb to generate all figures.')

events_processed.pkl saved  (3618 KB)
meta.json saved
has_360 = True

Run 02_visualizations.ipynb to generate all figures.
